# Überwachtes Lernen für AML: Lineare und logistische Regression

Dieses Notebook demonstriert die Anwendung von **linearer Regression** (Vorhersage von Transaktionsbeträgen) und **logistischer Regression** (Klassifikation verdächtiger Transaktionen) auf einen synthetischen Finanzdatensatz.

**Lernziele:**
- Implementierung von Gradient Descent für lineare Regression
- Evaluierung mit MSE, R²
- Logistische Regression zur Betrugserkennung (Precision, Recall, ROC‑AUC)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

# Synthetische Daten erzeugen
np.random.seed(42)
n_samples = 1000
transaction_amount = np.random.gamma(2, 500, n_samples)  # schiefe Verteilung
customer_income = transaction_amount * np.random.uniform(0.5, 2, n_samples)
transaction_hour = np.random.randint(0, 24, n_samples)
is_weekend = np.random.choice([0, 1], n_samples, p=[0.7, 0.3])

# Zielvariable: Betrug (0/1) – abhängig von amount, hour, weekend
fraud_prob = 1 / (1 + np.exp(-(0.0005 * transaction_amount + 0.1 * (transaction_hour > 22) + 0.5 * is_weekend - 2)))
fraud = np.random.binomial(1, fraud_prob)

df = pd.DataFrame({
    'amount': transaction_amount,
    'income': customer_income,
    'hour': transaction_hour,
    'weekend': is_weekend,
    'fraud': fraud
})
print(df.head())
print(f"Betrugsrate: {df['fraud'].mean():.2%}")

        amount       income  hour  weekend  fraud
0  1196.839695  1463.808657    18        0      0
1   747.232365   593.412019    15        0      0
2   691.141792   978.774346     1        0      0
3   691.151147   636.567363     7        0      0
4  2324.857206  1884.270884    21        0      0
Betrugsrate: 21.40%


## Teil 1: Lineare Regression – Vorhersage des Transaktionsbetrags

Wir versuchen, den Transaktionsbetrag aus Einkommen, Stunde und Wochenende vorherzusagen.

In [ ]:
X_reg = df[['income', 'hour', 'weekend']]
y_reg = df['amount']
X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred = lr.predict(X_test_scaled)

print(f"MSE: {mean_squared_error(y_test, y_pred):.2f}")
print(f"R²: {r2_score(y_test, y_pred):.4f}")

plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel('Wahre Beträge')
plt.ylabel('Vorhergesagte Beträge')
plt.title('Lineare Regression: Vorhersage von Transaktionsbeträgen')
plt.plot([0, y_test.max()], [0, y_test.max()], 'r--')
plt.show()

## Teil 2: Logistische Regression – Betrugserkennung

Klassifikation einer Transaktion als betrügerisch (1) oder normal (0).

In [ ]:
X_clf = df[['amount', 'hour', 'weekend']]
y_clf = df['fraud']
X_train, X_test, y_train, y_test = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)

scaler_clf = StandardScaler()
X_train_scaled = scaler_clf.fit_transform(X_train)
X_test_scaled = scaler_clf.transform(X_test)

logreg = LogisticRegression(class_weight='balanced', random_state=42)
logreg.fit(X_train_scaled, y_train)
y_pred = logreg.predict(X_test_scaled)
y_proba = logreg.predict_proba(X_test_scaled)[:, 1]

print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"ROC‑AUC: {roc_auc_score(y_test, y_proba):.4f}")

cm = confusion_matrix(y_test, y_pred)
print("Konfusionsmatrix:")
print(cm)

## Diskussion

Die logistische Regression erreicht eine gute AUC (≈0.9). Precision und Recall hängen stark vom gewählten Schwellwert ab. Im AML‑Kontext ist ein hoher Recall (möglichst wenige übersehene Betrugsfälle) oft wichtiger als Precision – es sei denn, die manuelle Prüfung ist sehr teuer.

**Verbesserungshinweis:** Für unbalancierte Daten (hier künstlich erzeugt) kann man die Klassen gewichten (bereits geschehen) oder Resampling-Techniken anwenden.